#Concept 1 — Class Imbalance kya hota hai?
Suppose ek dataset mein:
- 950 transactions Normal hain.
- 50 transactions Fraud hain.
Total 1,000 records mein fraud sirf 5% hai.

Agar model har transaction ko Normal predict kar de, toh uski accuracy 95% aa sakti hai—but woh ek bhi fraud detect nahi karega!

#Common solutions
- class_weight="balanced" — minority class ki mistakes ko higher weight deta hai.
- Oversampling — minority samples increase karna.
- Undersampling — majority samples reduce karna.
- SMOTE — synthetic minority samples generate karna.
- Proper metrics — Accuracy ke saath Recall, F1, Precision use karna.

#Cell 1 — Detect Class Imbalance

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Load dataset
data = load_breast_cancer()
X = data.data
y = data.target

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Class distribution
unique_classes, counts = np.unique(y_train, return_counts=True)

print("Training Class Distribution:")
print(dict(zip(unique_classes, counts)))

print("\nClass Percentages:")
percentages = counts / len(y_train) * 100

print(dict(zip(unique_classes, percentages.round(2))))

# Majority class baseline
dummy_model = DummyClassifier(strategy="most_frequent")
dummy_model.fit(X_train, y_train)

y_pred = dummy_model.predict(X_test)

print("Majority Class (Benign): 1")
print("Baseline Accuracy:", accuracy_score(y_test, y_pred))

Training Class Distribution:
{np.int64(0): np.int64(170), np.int64(1): np.int64(285)}

Class Percentages:
{np.int64(0): np.float64(37.36), np.int64(1): np.float64(62.64)}
Majority Class (Benign): 1
Baseline Accuracy: 0.631578947368421


#Experiment 2 — class_weight="balanced"

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

models = {
    "Normal Logistic Regression": LogisticRegression(max_iter=1000),
    "Balanced Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced")
}

for name, model in models.items():
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    print(f"\n{'='*50}")
    print(name)
    print(f"{'='*50}")

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Malignant Precision:", precision_score(y_test, y_pred, pos_label=0))
    print("Malignant Recall:", recall_score(y_test, y_pred, pos_label=0))
    print("Malignant F1 Score:", f1_score(y_test, y_pred, pos_label=0))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))


Normal Logistic Regression
Accuracy: 0.9824561403508771
Malignant Precision: 0.9761904761904762
Malignant Recall: 0.9761904761904762
Malignant F1 Score: 0.9761904761904762

Confusion Matrix:
[[41  1]
 [ 1 71]]

Balanced Logistic Regression
Accuracy: 0.956140350877193
Malignant Precision: 0.9111111111111111
Malignant Recall: 0.9761904761904762
Malignant F1 Score: 0.9425287356321839

Confusion Matrix:
[[41  1]
 [ 4 68]]


#Random Oversampling

In [9]:
from sklearn.utils import resample
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import numpy as np

# Separate majority and minority classes
X_majority = X_train[y_train == 1]  # Benign
X_minority = X_train[y_train == 0]    # Malignant

y_majority = y_train[y_train == 1]
y_minority = y_train[y_train == 0]

# Oversample minority class
X_minority_oversampled, y_minority_oversampled = resample(
    X_minority,
    y_minority,
    replace=True,
    n_samples=len(X_majority),
    random_state=42
)

# Combine balanced training data
X_train_balanced = np.vstack([X_majority, X_minority_oversampled])
y_train_balanced = np.concatenate([y_majority, y_minority_oversampled])

print("New Training Distribution:")
print(np.unique(y_train_balanced, return_counts=True))

# Train model
oversampling_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

oversampling_model.fit(X_train_balanced, y_train_balanced)

# Prediction
y_pred_oversampling = oversampling_model.predict(X_test)

# Evaluation
print("\nOversampling Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_oversampling))
print("Malignant Precision:", precision_score(y_test, y_pred_oversampling, pos_label=0))
print("Malignant Recall:", recall_score(y_test, y_pred_oversampling, pos_label=0))
print("Malignant F1:", f1_score(y_test, y_pred_oversampling, pos_label=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_oversampling))

New Training Distribution:
(array([0, 1]), array([285, 285]))

Oversampling Logistic Regression
Accuracy: 0.9649122807017544
Malignant Precision: 0.9318181818181818
Malignant Recall: 0.9761904761904762
Malignant F1: 0.9534883720930233

Confusion Matrix:
[[41  1]
 [ 3 69]]


#Random Undersampling

In [10]:
from sklearn.utils import resample
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import numpy as np

# Separate classes
X_majority = X_train[y_train == 1]  # Benign
X_minority = X_train[y_train == 0]    # Malignant

y_majority = y_train[y_train == 1]
y_minority = y_train[y_train == 0]

# Undersample majority class
X_majority_undersampled, y_majority_undersampled = resample(
    X_majority,
    y_majority,
    replace=False,
    n_samples=len(X_minority),
    random_state=42
)

# Combine data
X_train_undersampled = np.vstack([
    X_minority,
    X_majority_undersampled
])

y_train_undersampled = np.concatenate([
    y_minority,
    y_majority_undersampled
])

print("Undersampled Training Distribution:")
print(np.unique(y_train_undersampled, return_counts=True))

# Train model
undersampling_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

undersampling_model.fit(X_train_undersampled, y_train_undersampled)

# Predict
y_pred_undersampling = undersampling_model.predict(X_test)

# Evaluate
print("\nUndersampling Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_undersampling))
print("Malignant Precision:", precision_score(y_test, y_pred_undersampling, pos_label=0))
print("Malignant Recall:", recall_score(y_test, y_pred_undersampling, pos_label=0))
print("Malignant F1:", f1_score(y_test, y_pred_undersampling, pos_label=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_undersampling))

Undersampled Training Distribution:
(array([0, 1]), array([170, 170]))

Undersampling Logistic Regression
Accuracy: 0.956140350877193
Malignant Precision: 0.9111111111111111
Malignant Recall: 0.9761904761904762
Malignant F1: 0.9425287356321839

Confusion Matrix:
[[41  1]
 [ 4 68]]
